# Order lifecycle and account risk

A production backtest needs more than timestamped entries. Quotes are
amended, stale orders are cancelled, and account limits must reject unsafe
intent before it reaches the simulated venue. This recipe exercises that
complete lifecycle with stable client order IDs and inspects the resulting
audit trail.

In [1]:
import datetime as dt

import h5i_db
from h5i_db import backtest
import cookbook_utils as cu

INSTRUMENT_ID = "RATE-CUT-YES"
MARKET_CUT = "lifecycle-market-cut"
SECOND = 1_000_000_000

fixture = cu.make_backtest_fixture(steps=120, instrument_id=INSTRUMENT_ID)
db = h5i_db.Database(cu.fresh_db("06_order_lifecycle_and_risk"), create=True)
for name, table in fixture.items():
    db.create_table(name, table.schema, time_column="ts_init")
    db.append(name, table, note="deterministic lifecycle fixture")
db.snapshot(
    MARKET_CUT,
    tables=["instruments", "book_deltas", "trades"],
    note="Approved market-data cut for lifecycle examples",
)

{'name': 'lifecycle-market-cut',
 'created_at_ns': 1785367194944694214,
 'note': 'Approved market-data cut for lifecycle examples',
 'entries': {'38d5a925-3b36-4f8c-b08b-c3c8c7a8fe29': {'table_name': 'book_deltas',
   'sequence': 1,
   'manifest_checksum': '44c88aaa1135f661666e91984f10029b26c2db6b55abf17d45183a14336ee3a3'},
  'b841b6d9-0a75-43c7-8f3b-6d9c8a6429ba': {'table_name': 'instruments',
   'sequence': 1,
   'manifest_checksum': 'aa63921d42a312e03db7733c40074432bf1e84e1ae3ac40e814971f6917a7ec7'},
  'be7041cf-7df1-467f-8cd9-14d214e59901': {'table_name': 'trades',
   'sequence': 1,
   'manifest_checksum': '002021fcac63bdca193e5fbd43e2ce0ec1d78da9a34cfa535440257469369d74'}},
 'checksum': '90d7c4f0c884052b617a6fb241524733b06c338486457f707c1a7d0f05db1051'}

## Declare the lifecycle

`client_order_id` belongs to the strategy, not the engine. Later rows use
that stable name to address the exact order created by `submit`. Submit
fields are nullable in the storage schema because cancel rows only need an
ID; the builder validates the fields required by each action.

In [2]:
base = dt.datetime(2026, 6, 1, 14, 0, 0)
lifecycle = backtest.command_table(
    [
        {
            "ts": base + dt.timedelta(seconds=10),
            "action": "submit",
            "client_order_id": "yes-quote-001",
            "instrument_id": INSTRUMENT_ID,
            "side": "buy",
            "quantity": 20.0,
            "kind": "limit",
            "limit_price": 0.25,
            "tag": "passive-quote",
        },
        {
            "ts": base + dt.timedelta(seconds=30),
            "action": "amend",
            "client_order_id": "yes-quote-001",
            "quantity": 10.0,
            "limit_price": 0.26,
        },
        {
            "ts": base + dt.timedelta(seconds=60),
            "action": "cancel",
            "client_order_id": "yes-quote-001",
        },
    ]
)
backtest.create_command_table(db, "lifecycle_commands")
db.append(
    "lifecycle_commands",
    lifecycle,
    note="submit, reprice/resize, then cancel one quote",
)
lifecycle.to_pandas()

,ts,action,client_order_id,instrument_id,outcome,side,quantity,kind,limit_price,time_in_force,tag,reduce_only
0,2026-06-01 14:00:10,submit,yes-quote-001,RATE-CUT-YES,0.0,buy,20.0,limit,0.25,NaN,passive-quote,False
1,2026-06-01 14:00:30,amend,yes-quote-001,NaN,NaN,NaN,10.0,NaN,0.26,NaN,NaN,None
2,2026-06-01 14:01:00,cancel,yes-quote-001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


The typed configuration captures every material assumption. Preflight checks
the market pin, schemas, coverage, and the strongest fidelity supported by
the feed before the expensive replay starts.

In [3]:
lifecycle_config = backtest.BacktestConfig(
    run_id="lifecycle",
    portfolio=backtest.PortfolioConfig(starting_cash=10_000.0),
    data=backtest.DataConfig(
        commands="lifecycle_commands",
        snapshot=MARKET_CUT,
    ),
    execution=backtest.ExecutionConfig(
        fee_kind="prediction_market",
        fee_rate=0.02,
        latency_nanos=2_000_000,
    ),
    risk=backtest.RiskConfig(
        max_order_quantity=25.0,
        max_abs_position=50.0,
        max_open_orders=4,
    ),
    output=backtest.OutputConfig(equity_interval_nanos=5 * SECOND),
    metadata={"research_ticket": "PM-142", "owner": "market-making"},
)
inspection = backtest.inspect(db, lifecycle_config)
inspection.to_dict()

{'ok': True,
 'config_digest': '58983ab2fbcf3104c71021aa7a11c3526d48303c84c9d1aec835814aab6cb6ed',
 'fidelity': 'snapshot_l2',
 'tables': {'book_deltas': {'row_count': 240,
   'min_time': Timestamp('2026-06-01 14:00:01'),
   'max_time': Timestamp('2026-06-01 14:02:00'),
   'columns': ('ts_init',
    'ts_event',
    'instrument_id',
    'outcome',
    'action',
    'side',
    'price',
    'size',
    'event_index',
    'is_last',
    'source_vendor'),
   'actions': {'snapshot': 240}},
  'instruments': {'row_count': 2,
   'min_time': Timestamp('2026-06-01 14:00:00'),
   'max_time': Timestamp('2026-06-01 14:00:00'),
   'columns': ('ts_init',
    'instrument_id',
    'venue',
    'kind',
    'outcome',
    'outcome_label',
    'tick_size',
    'lot_size',
    'expiration_ns',
    'settlement_observable_ns')},
  'lifecycle_commands': {'row_count': 3,
   'min_time': Timestamp('2026-06-01 14:00:10'),
   'max_time': Timestamp('2026-06-01 14:01:00'),
   'columns': ('ts',
    'action',
    'cli

In [4]:
inspection.raise_for_errors()
lifecycle_result = backtest.execute(db, lifecycle_config)
orders = lifecycle_result.orders.to_pandas()
orders[
    [
        "order_id",
        "side",
        "limit_price",
        "quantity",
        "filled",
        "status",
        "reject_reason",
        "tag",
    ]
]

,order_id,side,limit_price,quantity,filled,status,reject_reason,tag
0,1,buy,0.26,10.0,0.0,cancelled,NaN,passive-quote


The quote was intentionally away from the market, so cancellation—not a
fill—is the expected outcome. `explain()` makes silence inspectable, and
`verify()` reruns the persisted config and compares every authoritative
output table.

In [5]:
assert lifecycle_result["fills"] == 0
assert orders["status"].tolist() == ["cancelled"]
explanation = lifecycle_result.explain()
verification = lifecycle_result.verify()
assert verification["verified"]
explanation

{'warnings': ["1 order(s) were cancelled without filling; the usual cause is acting before the instrument's first book update, or a limit the book never reached",
  '1 orders, no fills: 1 found no liquidity at their price'],
 'status_counts': {'cancelled': 1},
 'rejection_reasons': {},
 'orders_without_fills': 1,
 'orders_sweeping_multiple_levels': 0,
 'fidelity': 'snapshot_l2'}

## Prove that risk rejects before venue execution

Risk controls are native engine constraints, not notebook-side filters. The
oversized market order is recorded as rejected, never enters latency or
matching, and carries a durable reason in `bt_orders`.

In [6]:
risk_commands = backtest.command_table(
    [
        {
            "ts": base + dt.timedelta(seconds=20),
            "action": "submit",
            "client_order_id": "oversized-entry",
            "instrument_id": INSTRUMENT_ID,
            "side": "buy",
            "quantity": 100.0,
            "tag": "must-reject",
        }
    ]
)
backtest.create_command_table(db, "risk_commands")
db.append("risk_commands", risk_commands, note="risk rejection demonstration")

risk_config = backtest.BacktestConfig(
    run_id="risk-rejection",
    portfolio=backtest.PortfolioConfig(starting_cash=10_000.0),
    data=backtest.DataConfig(commands="risk_commands", snapshot=MARKET_CUT),
    risk=backtest.RiskConfig(
        max_order_quantity=25.0,
        max_abs_position=50.0,
        max_open_orders=4,
    ),
)
risk_result = backtest.execute(db, risk_config)
risk_order = risk_result.orders.to_pylist()[0]
assert risk_result["fills"] == 0
assert risk_order["status"] == "rejected"
assert "max_order_quantity" in risk_order["reject_reason"]
risk_result.explain()

{'warnings': ['1 orders, no fills: 1 refused by configured risk limits'],
 'status_counts': {'rejected': 1},
 'rejection_reasons': {'quantity 100 exceeds max_order_quantity 25': 1},
 'orders_without_fills': 1,
 'orders_sweeping_multiple_levels': 0,
 'fidelity': 'snapshot_l2'}

## Takeaways

- Stable client IDs make amend/cancel workflows independent of engine IDs.
- Amendments follow venue-like queue rules; repricing or increasing size
  loses priority.
- Position limits include all live orders, not only already-filled exposure.
- Rejection reasons are persisted and queryable, so a zero-fill run is
  diagnosable.
- Typed configs, preflight, and semantic verification form one reproducible
  operational contract.

In [7]:
db.close()